# 01 - Search-space definition

This notebook confirms that the tunable search spaces declared on the model
configuration classes are well formed and can be enumerated. It exercises
`configuration.models_config` (the `tunable_lr_params` and `tunable_arch_params`
class methods) and shows, per phase, which parameters are sampled and over what
ranges or choice sets.

No tuning is run here; the goal is to make the declared search space visible and
verify by eye that every entry has a recognised sampling `type`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt
import optuna

import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    "figure.figsize" : (7.0, 4.2),
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
})


## Load the declared spaces

Phase 1 tunes optimisation parameters (learning rates, weight decays, dropout)
via `tunable_lr_params`. Phase 2 tunes architecture parameters via
`tunable_arch_params`. We read both for a representative model.

In [ ]:
from configuration.models_config import UNetConfig

lr_space   = UNetConfig.tunable_lr_params()
arch_space = UNetConfig.tunable_arch_params()

print('Phase-1 (lr) parameters :', len(lr_space))
for name, spec in lr_space.items():
    print(f'  {name:18s} {spec}')

print()
print('Phase-2 (arch) parameters:', len(arch_space))
for name, spec in arch_space.items():
    print(f'  {name:18s} {spec}')

## Validate every spec has a recognised type

The `ParamSampler` only understands three kinds of specification: `float`,
`categorical` and `indexed_categorical`. Any other type would be silently
skipped, so we assert the declared spaces use only these.

In [ ]:
VALID_TYPES = {'float', 'categorical', 'indexed_categorical'}

for phase, space in [('phase1', lr_space), ('phase2', arch_space)]:
    for name, spec in space.items():
        kind = spec['type']
        assert kind in VALID_TYPES, f'{phase}:{name} has unknown type {kind}'
        if kind == 'float':
            assert spec['low'] < spec['high'], f'{phase}:{name} low >= high'
        else:
            assert len(spec['choices']) > 0, f'{phase}:{name} empty choices'

print('All specifications use a recognised sampling type.')

## Visualise float ranges (log scale)

The learning-rate and weight-decay parameters are sampled in log space.
Plotting their declared `[low, high]` intervals on a log axis confirms the
spans the tuner will explore.

In [ ]:
float_specs = {k: v for k, v in lr_space.items() if v['type'] == 'float'}

fig, ax = plt.subplots()
for i, (name, spec) in enumerate(float_specs.items()):
    ax.plot([spec['low'], spec['high']], [i, i], marker='|', markersize=14, linewidth=2)
    ax.text(spec['high'], i, f"  {'log' if spec.get('log') else 'lin'}", va='center', fontsize=8)

ax.set_yticks(range(len(float_specs)))
ax.set_yticklabels(list(float_specs.keys()))
ax.set_xscale('log')
ax.set_xlabel('value')
ax.set_title('Phase-1 float search ranges (UNetConfig)')
fig.tight_layout()
plt.show()

## Visualise categorical cardinality

Each architecture parameter is a finite choice set. The bar chart shows how
many alternatives the tuner can pick from for each.

In [ ]:
names  = list(arch_space.keys())
counts = [len(arch_space[n]['choices']) for n in names]

fig, ax = plt.subplots()
ax.bar(names, counts, color='steelblue')
ax.set_ylabel('number of choices')
ax.set_title('Phase-2 categorical cardinality (UNetConfig)')
ax.tick_params(axis='x', rotation=30)
for i, c in enumerate(counts):
    ax.text(i, c + 0.05, str(c), ha='center', fontsize=9)
fig.tight_layout()
plt.show()

## Expected visual outcome

The type-validation cell prints a single success line with no assertion error.
The float-range plot shows nine horizontal spans on a log x-axis (the eight
lr/wd parameters plus dropout, the latter near the linear lower decade). The
cardinality bar chart shows five bars whose heights match the choice-set sizes
declared in `tunable_arch_params`.